# OpenSearch Vector Store With llama-index-pydocker

OpenSearch combines full-text search and approximate nearest-neighbour vector search in one engine. It is the common backend for hybrid search pipelines and enterprise search applications that need both lexical and semantic retrieval.

This notebook shows how `llama-index-pydocker` provisions a local OpenSearch container automatically, then demonstrates indexing, retrieval, and teardown.

## Table Of Contents

- [Prerequisites](#prerequisites)
- [1. Start OpenSearch With One Import Swap](#1-start-opensearch-with-one-import-swap)
- [2. Index Documents For Retrieval](#2-index-documents-for-retrieval)
- [3. Run And Validate Retrieval](#3-run-and-validate-retrieval)
- [4. Context Manager Teardown](#4-context-manager-teardown)
- [5. Remote Passthrough (No Docker)](#5-remote-passthrough-no-docker)
- [6. Conclusion](#6-conclusion)

## Prerequisites

- Docker Desktop must be running.
- Install dependencies:
  ```bash
  pip install "llama-index-pydocker[opensearch]"
  ```
- No environment variables required.
- Note: the OpenSearch image is large (~1 GB) and takes 1–2 minutes to become ready on first pull.

## 1. Start OpenSearch With One Import Swap

Replace `llama_index.vector_stores.opensearch` with `llama_index_pydocker`. The constructor adds `embed_dim` (passed to `OpensearchVectorClient`) and an optional `docker_config`. The container starts before the client connects.

In [ ]:
import sys
import uuid
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path().cwd().parent))

from llama_index_pydocker import OpensearchVectorStore
from docker_db import OpenSearchConfig

In [ ]:
temp_dir = Path(tempfile.mkdtemp())
container_name = f"demo-opensearch-{uuid.uuid4().hex[:8]}"

cfg = OpenSearchConfig(
    project_name="demo",
    container_name=container_name,
    volume_path=temp_dir / "osdata",
    retries=30,
    delay=3,
)

store = OpensearchVectorStore(
    index_name="demo_index",
    endpoint=f"http://localhost:{cfg.port}",
    embed_dim=384,
    docker_config=cfg,
)

print(f"OpenSearch started: {container_name}")
print(f"Index:              demo_index")

## 2. Index Documents For Retrieval

Pass `store` to `StorageContext`. LlamaIndex writes embeddings into the OpenSearch index.

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.embeddings import MockEmbedding
from llama_index.core.schema import Document

documents = [
    Document(text="OpenSearch supports approximate nearest-neighbour search using the k-NN plugin."),
    Document(text="Hybrid search in OpenSearch combines BM25 lexical scoring with vector similarity."),
    Document(text="Docker makes local infrastructure reproducible for development and testing."),
]

embed_model = MockEmbedding(embed_dim=384)
storage_context = StorageContext.from_defaults(vector_store=store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=False,
)

print(f"Indexed {len(documents)} documents.")

## 3. Run And Validate Retrieval

Retrieve nodes from OpenSearch to confirm the full data path is working.

In [ ]:
query = "How does OpenSearch handle vector search?"
retriever = index.as_retriever(similarity_top_k=2)
results = retriever.retrieve(query)

print(f"Retrieved {len(results)} nodes")
for i, node in enumerate(results, start=1):
    print(f"\nResult {i} (score={node.score:.4f})")
    print(node.text)

assert len(results) > 0

## 4. Context Manager Teardown

Call `store.stop()` to remove the container, or wrap the workflow in `with` for automatic cleanup.

In [ ]:
store.stop()
print(f"Container '{container_name}' removed.")

In [ ]:
# Equivalent pattern with automatic teardown
new_name = f"demo-opensearch-{uuid.uuid4().hex[:8]}"
new_cfg = cfg.model_copy(update={"container_name": new_name,
                                   "volume_path": temp_dir / new_name})

with OpensearchVectorStore(
    index_name="demo_index",
    endpoint=f"http://localhost:{new_cfg.port}",
    embed_dim=384,
    docker_config=new_cfg,
) as s:
    print(f"Container up: {s._db.config.container_name}")
print("Container removed.")

## 5. Remote Passthrough (No Docker)

Point the store at an Amazon OpenSearch Service endpoint and the wrapper is completely transparent.

In [ ]:
# Uncomment to target a hosted OpenSearch cluster — Docker is never started
#
# store = OpensearchVectorStore(
#     index_name="docs",
#     endpoint="https://my-cluster.us-east-1.es.amazonaws.com",
#     embed_dim=1536,
# )
# assert store._db is None   # no container

print("Remote passthrough: Docker is never touched for non-localhost URLs.")

## 6. Conclusion

You have completed the workflow introduced at the top:
1. Started an OpenSearch container with a single import swap.
2. Indexed documents through `OpensearchVectorStore` without managing the client manually.
3. Retrieved nodes and confirmed the data path is working end to end.
4. Cleaned up with `stop()` and with a context manager.

Key next steps:
- Replace `MockEmbedding` with a real embedding model and update `embed_dim` to match.
- Pass `engine='faiss'` or `engine='nmslib'` to `OpensearchVectorStore` to control the k-NN algorithm.
- Switch `endpoint` to an Amazon OpenSearch Service URL to deploy without any other code changes.